# dim_regions
- dim_region 
- Source table: pondolens.dev_silver.grdp_per_capita

In [0]:
CREATE OR REPLACE TABLE pondolens.dev_gold.dim_regions
USING DELTA
AS

WITH standardized_regions AS (
    SELECT DISTINCT
        REPLACE(
            UPPER(TRIM(region_code)),
            '-',
            ''
        ) AS region_code,

        UPPER(TRIM(region_name)) AS region,

        LOWER(TRIM(geographic_level)) AS geographic_level

    FROM pondolens.dev_silver.grdp_per_capita

        WHERE region_name IS NOT NULL
         AND TRIM(region_name) <> ''
         AND region_code IS NOT NULL
         AND TRIM(region_code) <> ''
         AND LOWER(TRIM(geographic_level)) = 'region'
         AND UPPER(TRIM(region_name)) <> 'PHILIPPINES'
        AND UPPER(TRIM(region_code)) NOT IN ('PH', 'PHL')
),

deduplicated_regions AS (
    SELECT
        region_code,
        region,
        geographic_level,

        ROW_NUMBER() OVER (
            PARTITION BY region_code
            ORDER BY region
        ) AS rn

    FROM standardized_regions
)

SELECT
    10000 + ROW_NUMBER() OVER (
        ORDER BY region_code
    ) AS region_id,

    region_code,
    region,
    geographic_level

FROM deduplicated_regions
WHERE rn = 1;

In [0]:
SELECT *
FROM pondolens.dev_gold.dim_regions